# 프로젝트 개요

**목표:** 서울의 실제 날씨 데이터를 API로 수집하고, Oracle DB의 배달 주문 데이터와 결합해서 "비 오는 날엔 어떤 메뉴가 더 잘 팔리는지" 분석하고, 그 결과를 Gradio 화면으로 출력

---

# 필수 스킬

| 단계 | 사용 기술 | 포인트 |
| :--- | :--- | :--- |
| 1. 날씨 데이터 수집 | `requests`로 외부 API 호출 (Open-Meteo, 무료·키 불필요) | 신규 — API 크롤링 |
| 2. 배달 주문 데이터 조회 | Oracle DB 연결 (`.env` + `with` 구문) | Chapter02 ex01 |
| 3. 데이터 결합 | `pd.merge()`, `groupby()`, `pivot_table()` | Chapter01, Chapter02 ex09 |
| 4. 통계 분석 | `numpy`로 상관관계, 그룹별 평균 비교 | Chapter02 ex02, ex10 |
| 5. 파일 저장 | `to_csv()`로 중간/최종 결과 저장 | 신규 (파일 저장 습관) |
| 6. 시각화 | `matplotlib` + `seaborn` (색상/라벨 커스터마이징) | Chapter02 ex03~06 |
| 7. 화면 구현 | `Gradio` 기반 UI 구축 | 신규 |

In [9]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://historical-forecast-api.open-meteo.com/v1/forecast"
params = {
	"latitude": 37.5665,
	"longitude": 126.9780,
	"daily": "weather_code",
	"models": "kma_seamless",
	"timezone": "Asia/Tokyo",
	"start_date": "2026-08-01",
	"end_date": "2026-08-31",
}
responses = openmeteo.weather_api(url, params = params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone: {response.Timezone()}{response.TimezoneAbbreviation()}")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process daily data. The order of variables needs to be the same as requested.
daily = response.Daily()
daily_weather_code = daily.Variables(0).ValuesAsNumpy()

daily_data = {
	"date": pd.date_range(
		start = pd.to_datetime(daily.Time(), unit = "s", utc = True),
		end =  pd.to_datetime(daily.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = daily.Interval()),
		inclusive = "left"
	).tz_convert(response.Timezone().decode())
}

daily_data["weather_code"] = daily_weather_code

daily_dataframe = pd.DataFrame(data = daily_data)
print("\nDaily data\n", daily_dataframe)


Coordinates: 37.56278991699219°N 126.9830322265625°E
Elevation: 34.0 m asl
Timezone: b'Asia/Tokyo'b'GMT+9'
Timezone difference to GMT+0: 32400s

Daily data
                         date  weather_code
0  2026-08-01 00:00:00+09:00           NaN
1  2026-08-02 00:00:00+09:00           NaN
2  2026-08-03 00:00:00+09:00           NaN
3  2026-08-04 00:00:00+09:00           NaN
4  2026-08-05 00:00:00+09:00           NaN
5  2026-08-06 00:00:00+09:00           NaN
6  2026-08-07 00:00:00+09:00           NaN
7  2026-08-08 00:00:00+09:00           NaN
8  2026-08-09 00:00:00+09:00           NaN
9  2026-08-10 00:00:00+09:00           NaN
10 2026-08-11 00:00:00+09:00           NaN
11 2026-08-12 00:00:00+09:00           NaN
12 2026-08-13 00:00:00+09:00           NaN
13 2026-08-14 00:00:00+09:00           NaN
14 2026-08-15 00:00:00+09:00           NaN
15 2026-08-16 00:00:00+09:00           NaN
16 2026-08-17 00:00:00+09:00           NaN
17 2026-08-18 00:00:00+09:00           NaN
18 2026-08-19 00:00:00+09:

In [ ]:
import os
from dotenv import load_dotenv
import oracledb
import pandas as pd

load_dotenv()
USER = os.getenv("ORACLE_USER")
PASSWORD = os.getenv("ORACLE_PASSWORD")
DSN = os.getenv("ORACLE_DSN")

with oracledb.connect(user=USER, password=PASSWORD, dsn=DSN) as conn:
    df=pd.read_sql("SELECT * FROM DELIVERY_ORDERS", conn)
# with문 블록을 벗어나는 순간 자동으로 conn.close()가 호출되어 안정적임

print("연결 및 쿼리 성공! 데이터 shape:", df.shape)
df.head()